# 01 · Data Check — Jena Climate (Seq2Seq Weather Forecasting)

## 1. Project & Dataset Overview

**Project.** Sequence-to-sequence weather forecasting at the Jena climate
station. Input is a **multivariate** 168-hour (7-day) window; output is a
**univariate** 72-hour (3-day) forecast of `T (degC)`. Downstream models
(Baseline, Seq2Seq LSTM, Attention LSTM, Transformer) are trained and
evaluated elsewhere — **this notebook does not train anything**.

**Dataset.** Jena Climate (Kaggle slug `mnassrib/jena-climate`), raw file
`data/raw/jena_climate_2009_2016.csv`, 10-minute-resolution weather
observations from the Max Planck Institute for Biogeochemistry weather
station in Jena, Germany.

| Item | Value |
|---|---|
| Target | `T (degC)` |
| Raw frequency | ~10 minutes |
| Modeling resolution | 1 hour (after resampling) |
| Input window | 168 hours |
| Forecast horizon | 72 hours |
| Split | 70% train / 15% validation / 15% test, chronological |
| Task | Multivariate input → univariate multi-step forecast |

**Purpose of this notebook.** Data-quality gate, exploratory data analysis,
and a written record of preprocessing *decisions* (hourly aggregation,
missing-data policy, split policy, feature schema). It does **not** modify
`data/raw/`, does **not** fit any scaler/imputer, and does **not** produce
`data/processed/`. All numbers below are computed live from the raw CSV —
none are hard-coded from the task brief.

**Ground rules enforced throughout this notebook**

- The raw CSV is read-only and is never overwritten.
- No row is dropped, no value is imputed, and no feature is scaled before
  its policy is explicitly decided in the relevant section.
- Validation/duplicate/gap detection logic reuses
  [`src/data/validator.py`](../src/data/validator.py) (`validate_csv`)
  instead of re-implementing a second pipeline.
- No absolute paths; everything is resolved relative to the project root
  with `pathlib`.
- Large series (420k raw rows) are downsampled/aggregated for plotting only,
  and every such plot says so explicitly.

## 2. Data Pipeline Overview

This notebook covers **Validate → Duplicate/Gap Check** rigorously and
records the decisions for **Hourly Resample → Feature Engineering →
Temporal Split → Train-only Missing/Scaler → Sliding Windows** without
writing processed data. Production transformations remain in the TV2 source
modules and are not reimplemented here.


## 3. Imports & Project Paths

Project root is located by walking upward from the notebook's working
directory until `pyproject.toml` and `src/` are both found — no hard-coded
absolute path, so `Restart & Run All` works regardless of where Jupyter was
launched from.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError(f"Could not locate project root starting from {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "jena_climate_2009_2016.csv"
PREPROCESSING_DIR = PROJECT_ROOT / "artifacts" / "preprocessing"
CLEANING_POLICY_PATH = PREPROCESSING_DIR / "tv2_data_cleaning_policy.json"
LOCKED_SCHEMA_PATH = PREPROCESSING_DIR / "feature_schema.json"

from src.data.validator import (  # noqa: E402
    EXPECTED_COLUMNS,
    EXPECTED_FREQUENCY_MINUTES,
    TARGET_COLUMN,
    TIMESTAMP_COLUMN,
    validate_csv,
)

numeric_columns = [c for c in EXPECTED_COLUMNS if c != TIMESTAMP_COLUMN]

INPUT_WINDOW_HOURS = 168
FORECAST_HORIZON_HOURS = 72
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
assert abs((TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION) - 1.0) < 1e-9

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

print("Project root  :", PROJECT_ROOT)
print("Raw CSV exists:", RAW_CSV_PATH.exists(), "->", RAW_CSV_PATH)


## 4. Load Raw Dataset

Two read-only views of the raw CSV are kept for the rest of the notebook:

- `df_raw` — exactly as `pandas.read_csv` returns it (original row order,
  original dtypes, `Date Time` still a string). This is what schema/type
  checks run against.
- `df_parsed` — the same rows with `Date Time` parsed to `datetime64`
  (`errors="coerce"` so parse failures surface as `NaT` instead of raising).
  Row order is **not** changed and duplicates are **not** dropped here —
  parsing a column is not a cleaning decision.

Neither frame is ever written back to `data/raw/`.

In [ ]:
df_raw = pd.read_csv(RAW_CSV_PATH)

df_parsed = df_raw.copy()
df_parsed[TIMESTAMP_COLUMN] = pd.to_datetime(
    df_raw[TIMESTAMP_COLUMN], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)

print("df_raw shape   :", df_raw.shape)
print("df_parsed shape:", df_parsed.shape)
df_raw.head(3)

## 5. Dataset Summary

In [ ]:
summary = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "n_unique": df_raw.nunique(),
        "n_missing": df_raw.isna().sum(),
    }
)
print(f"Rows: {len(df_raw):,}  |  Columns: {df_raw.shape[1]}")
summary

## 6. Schema Validation

Reuses `src.data.validator.validate_csv`, the single production validation
entry point, instead of re-checking columns with ad-hoc notebook code. The
full report is kept in `report` and referenced throughout the notebook.

In [ ]:
report = validate_csv(RAW_CSV_PATH)

schema_table = pd.DataFrame(
    {
        "check": [
            "row_count",
            "column_count",
            "missing_columns",
            "unexpected_columns",
            "columns_match_expected_order",
            "target_column_exists",
        ],
        "value": [
            report["row_count"],
            report["column_count"],
            report["missing_columns"] or "-",
            report["unexpected_columns"] or "-",
            report["schema_valid"],
            report["target_column_exists"],
        ],
    }
)
schema_status = "PASS" if (report["schema_valid"] and report["target_column_exists"]) else "FAIL"
print("Schema status:", schema_status)
schema_table

**Interpretation.** `schema_valid=True` means the 15 raw columns match
`EXPECTED_COLUMNS` in name, count, and order exactly, and the target column
`T (degC)` is present. Schema gate: **PASS**.

## 7. Timestamp Quality Check

All figures below come straight out of `report["timestamp"]`
(`validate_csv`'s own logic) — nothing here re-derives parsing, monotonicity
or gap detection.

In [ ]:
ts_report = report["timestamp"]

ts_summary = pd.DataFrame(
    {
        "field": [
            "format",
            "parse_errors",
            "date_min",
            "date_max",
            "monotonic_in_file",
            "duplicate_rows_after_first",
            "expected_frequency_minutes",
            "mode_interval_minutes",
            "gap_count",
        ],
        "value": [
            ts_report["format"],
            ts_report["parse_errors"],
            ts_report["date_min"],
            ts_report["date_max"],
            ts_report["monotonic_in_file"],
            ts_report["duplicate_rows_after_first"],
            ts_report["expected_frequency_minutes"],
            ts_report["mode_interval_minutes"],
            ts_report["gap_count"],
        ],
    }
)
ts_summary

In [ ]:
# Interval distribution over unique, sorted timestamps (diagnostic only —
# reuses the same dedup+sort approach validate_csv takes internally before
# diffing, it does not persist or alter df_parsed).
unique_sorted_ts = (
    df_parsed[TIMESTAMP_COLUMN].dropna().drop_duplicates().sort_values().reset_index(drop=True)
)
interval_minutes = unique_sorted_ts.diff().dropna().dt.total_seconds() / 60

interval_counts = interval_minutes.value_counts().sort_index()
top_intervals = interval_counts.sort_values(ascending=False).head(8).rename("count").to_frame()
top_intervals["pct_of_intervals"] = (top_intervals["count"] / len(interval_minutes) * 100).round(4)
top_intervals.index.name = "interval_minutes"
top_intervals

**Interpretation.** The dominant interval accounts for the vast majority of
consecutive-timestamp gaps and matches `mode_interval_minutes` above,
confirming the raw ~10-minute cadence. Any interval `> 10` minutes is a
temporal gap (analyzed in §9); `monotonic_in_file=False` means the raw file
is **not** strictly ordered by time even after accounting for exact repeats
— investigated next (§8) as it lines up with the duplicate timestamp
ranges.

## 8. Duplicate Timestamp Analysis

The validator only reports a single count (`duplicate_rows_after_first`).
Production code has no equivalent for **classifying** duplicates, so this
section adds that analysis on top of (not instead of) the validator's
count, and cross-checks the totals against it:

- **A. Exact full-row duplicate** — every feature column is identical
  across all rows sharing a timestamp.
- **B. Conflicting duplicate** — same timestamp, but at least one feature
  column differs between the rows.

In [ ]:
feature_columns = [c for c in df_parsed.columns if c != TIMESTAMP_COLUMN]

dup_mask = df_parsed[TIMESTAMP_COLUMN].duplicated(keep=False) & df_parsed[TIMESTAMP_COLUMN].notna()
dup_rows = df_parsed[dup_mask]

records = []
for ts, group in dup_rows.groupby(TIMESTAMP_COLUMN):
    n_distinct_rows = group[feature_columns].drop_duplicates().shape[0]
    records.append(
        {
            "Date Time": ts,
            "occurrence_count": len(group),
            "distinct_feature_rows": n_distinct_rows,
            "is_exact_duplicate": n_distinct_rows == 1,
            "row_indices": list(group.index),
        }
    )

duplicate_timestamps_df = pd.DataFrame.from_records(records).sort_values("Date Time").reset_index(drop=True)

n_unique_dup_timestamps = len(duplicate_timestamps_df)
n_exact_groups = int(duplicate_timestamps_df["is_exact_duplicate"].sum())
n_conflicting_groups = n_unique_dup_timestamps - n_exact_groups
rows_after_first_computed = int((duplicate_timestamps_df["occurrence_count"] - 1).sum())

dup_summary = pd.DataFrame(
    {
        "metric": [
            "unique_duplicate_timestamps",
            "exact_duplicate_groups (A)",
            "conflicting_duplicate_groups (B)",
            "duplicate_rows_after_first (computed)",
            "duplicate_rows_after_first (validator report)",
            "matches_validator_report",
        ],
        "value": [
            n_unique_dup_timestamps,
            n_exact_groups,
            n_conflicting_groups,
            rows_after_first_computed,
            ts_report["duplicate_rows_after_first"],
            rows_after_first_computed == ts_report["duplicate_rows_after_first"],
        ],
    }
)
dup_summary

**Interpretation.** If `conflicting_duplicate_groups (B) == 0`, every
duplicate timestamp in the raw file carries identical feature values, so
resolving duplicates by keeping the first occurrence per timestamp (the
approved policy in `tv2_data_cleaning_policy.json`) discards no information.
If B were `> 0`, a first-row-wins policy would silently pick one of several
disagreeing observations and that risk would need to be called out
explicitly in the final follow-up.

## 9. Temporal Gap Analysis

Built directly from `report["timestamp"]["gaps"]` (the validator's own gap
list) — this section only adds `estimated_missing_intervals` and a
`severity` bucket on top of it, it does not re-detect gaps independently.

In [ ]:
def classify_gap_severity(interval_minutes: float) -> str:
    hours = interval_minutes / 60
    if hours < 1:
        return "minor"
    if hours < 6:
        return "moderate"
    if hours < 24:
        return "major"
    return "critical"


gap_records = ts_report["gaps"]
gap_df = pd.DataFrame(gap_records)
if not gap_df.empty:
    # validator's own field names are the reverse of the intuitive reading:
    # its "after" holds the last reading *before* the gap, and its "before"
    # holds the first reading *after* the gap. Relabel once to the requested
    # before/after convention (before = last reading before the gap,
    # after = first reading after it) and leave the values untouched.
    gap_df = gap_df.rename(columns={"after": "before", "before": "after"})
    gap_df["estimated_missing_intervals"] = (
        gap_df["interval_minutes"] / EXPECTED_FREQUENCY_MINUTES - 1
    ).round().astype(int)
    gap_df["severity"] = gap_df["interval_minutes"].apply(classify_gap_severity)
    gap_df = gap_df[["before", "after", "interval_minutes", "estimated_missing_intervals", "severity"]]

print(f"gap_count (validator) = {ts_report['gap_count']}  |  rows built = {len(gap_df)}")
gap_df

In [ ]:
from matplotlib import dates as mdates
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(11, 2.2))
ts_min = float(mdates.date2num(unique_sorted_ts.iloc[0].to_pydatetime()))
ts_max = float(mdates.date2num(unique_sorted_ts.iloc[-1].to_pydatetime()))
ax.hlines(y=0, xmin=ts_min, xmax=ts_max, color="#93c5fd", linewidth=4)
for _, row in gap_df.iterrows():
    gap_start = float(mdates.date2num(pd.Timestamp(str(row["before"])).to_pydatetime()))
    gap_end = float(mdates.date2num(pd.Timestamp(str(row["after"])).to_pydatetime()))
    ax.axvspan(gap_start, gap_end, color="#dc2626", alpha=0.6)
gap_legend = Patch(facecolor="#dc2626", alpha=0.6, label="temporal gap (>10 min)")
ax.set_yticks([])
ax.set_xlabel("Date")
ax.set_title(f"Raw timestamp coverage 2009-2016 ({len(gap_df)} gap(s) highlighted)")
ax.legend(handles=[gap_legend], loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


**Interpretation.** All gaps are `interval_minutes > 10`, i.e. missing raw
10-minute readings, not missing rows with `NaN` — the raw file simply has no
record for that time window. The two `critical` gaps (`> 24h`) are the
tightest constraint on the sliding-window contract (§26): no 168h input
window or 72h horizon window may straddle one of these.

## 10. Missing Value Analysis

From `report["missing_by_column"]` (validator output) — the count of `NaN`
cells actually present in the raw CSV, which is distinct from the temporal
gaps in §9 (a gap is a missing *row*; this section is about missing
*cells* within rows that do exist).

In [ ]:
missing_table = pd.DataFrame(
    {
        "missing_count": pd.Series(report["missing_by_column"]),
    }
)
missing_table["missing_percentage"] = (missing_table["missing_count"] / report["row_count"] * 100).round(4)
missing_status = "PASS" if missing_table["missing_count"].sum() == 0 else "WARNING"
print("Total missing cells:", int(missing_table["missing_count"].sum()), "| status:", missing_status)
missing_table

## 11. Non-numeric / Infinite Check

First reuses the validator's `non_numeric_by_column` / `infinite_by_column`
(structural checks). Then adds a **domain plausibility check** that the
validator does not perform: a value can be perfectly numeric and finite yet
still be physically impossible (e.g. a negative wind speed). This is new
analysis, not a duplicate of any existing pipeline.

In [ ]:
non_numeric_inf_table = pd.DataFrame(
    {
        "non_numeric_count": pd.Series(report["non_numeric_by_column"]),
        "infinite_count": pd.Series(report["infinite_by_column"]),
    }
)
struct_status = "PASS" if non_numeric_inf_table.to_numpy().sum() == 0 else "FAIL"
print("Non-numeric/Infinite status:", struct_status)
non_numeric_inf_table

In [ ]:
# Physically plausible ranges, from domain knowledge of the measured
# quantities (not fitted from data): relative humidity in [0,100]%, wind
# direction in [0,360] deg, several quantities are non-negative by
# definition, and station-level air pressure has a broad but bounded range.
PLAUSIBLE_RANGES = {
    "p (mbar)": (800.0, 1100.0),
    "rh (%)": (0.0, 100.0),
    "VPmax (mbar)": (0.0, None),
    "VPact (mbar)": (0.0, None),
    "VPdef (mbar)": (0.0, None),
    "sh (g/kg)": (0.0, None),
    "H2OC (mmol/mol)": (0.0, None),
    "rho (g/m**3)": (0.0, None),
    "wv (m/s)": (0.0, None),
    "max. wv (m/s)": (0.0, None),
    "wd (deg)": (0.0, 360.0),
}

plausibility_records = []
for column, (lower, upper) in PLAUSIBLE_RANGES.items():
    series: pd.Series = pd.Series(
        pd.to_numeric(df_raw[column], errors="coerce"),
        index=df_raw.index,
        dtype="float64",
    )
    below_count = int((series < lower).sum()) if lower is not None else 0
    above_count = int((series > upper).sum()) if upper is not None else 0
    plausibility_records.append(
        {
            "column": column,
            "expected_range": f"[{lower if lower is not None else '-inf'}, {upper if upper is not None else '+inf'}]",
            "violations_below": below_count,
            "violations_above": above_count,
            "total_violations": below_count + above_count,
            "observed_min": float(series.min()),
            "observed_max": float(series.max()),
        }
    )

plausibility_df = pd.DataFrame.from_records(plausibility_records).sort_values(
    "total_violations", ascending=False
).reset_index(drop=True)
plausibility_status = "PASS" if plausibility_df["total_violations"].sum() == 0 else "FAIL"
print("Domain plausibility status:", plausibility_status)
plausibility_df

**Interpretation.** `wv (m/s)` and `max. wv (m/s)` contain observed minima
far below any physically valid wind speed — a well-known sentinel/error
code (`-9999`) in this dataset used by the original logger for missing or
faulty wind readings. The validator's structural checks (non-numeric,
infinite, `NaN`) all report **zero** issues for these columns because
`-9999.0` is a perfectly valid finite float — this plausibility check is the
only thing in the pipeline that catches it. This is a real data-quality
finding that the approved `tv2_data_cleaning_policy.json` does not yet
address; it is carried into §24 and §29 as a required action, not silently
fixed here.

## 12. Constant / Near-constant Feature Check

`constant_columns` (exact constants, `nunique<=1`) comes from the
validator. Near-constant is new: a numeric column whose spread is tiny
relative to its own *typical* range. Std is normalized by the robust P1-P99
range rather than the mean, because these features are interval-scaled with
a large, arbitrary offset (e.g. pressure in mbar, temperature in degC) —
dividing by the mean would flag any large-offset feature as "near-constant"
regardless of its real variability, and dividing by the raw min-max range
would instead be distorted by the `-9999` sentinel outliers found in §11.

In [ ]:
numeric_df = df_raw[numeric_columns].apply(pd.to_numeric, errors="coerce")

p1: pd.Series = numeric_df.quantile(0.01)
p99: pd.Series = numeric_df.quantile(0.99)
robust_range: pd.Series = (p99 - p1).replace(0, np.nan)

variance_table = pd.DataFrame(
    {
        "n_unique": numeric_df.nunique(),
        "std": numeric_df.std(),
        "p1_p99_range": robust_range,
    }
)
variance_table["std_over_robust_range"] = variance_table["std"] / variance_table["p1_p99_range"]
variance_table["is_constant"] = variance_table.index.isin(report["constant_columns"])
NEAR_CONSTANT_RATIO_THRESHOLD = 0.02
variance_table["is_near_constant"] = (
    variance_table["std_over_robust_range"] < NEAR_CONSTANT_RATIO_THRESHOLD
) & (~variance_table["is_constant"])

constant_status = "PASS" if not variance_table["is_constant"].any() else "WARNING"
print("Constant columns (validator):", report["constant_columns"] or "none")
print("Near-constant columns (std/P1-P99-range < %.2f):" % NEAR_CONSTANT_RATIO_THRESHOLD,
      list(variance_table.index[variance_table["is_near_constant"]]) or "none")
variance_table.round(4)

## 13. Descriptive Statistics

Standard five-number summary plus mean/std for every raw numeric feature,
computed on the full 420k-row dataset (a table, not a data dump).

In [ ]:
describe_table = numeric_df.describe().T
describe_table["skew"] = numeric_df.skew()
describe_table.round(3)

**Interpretation.** `wv (m/s)` and `max. wv (m/s)` show `min = -9999.00`,
confirming §11's plausibility finding directly in the summary statistics —
it also drags their `mean` far below their `50%` (median), a classic sentinel-value
signature. All other columns have min/max consistent with plausible
station-level meteorological readings.

## 14. Target Analysis – T (degC)

Time-series and distribution views of the forecasting target. The
"over time" plot is **downsampled to a daily mean** purely for
readability — 420k raw points would render as an unreadable smear; the
underlying data used for every other check in this notebook is untouched.

In [ ]:
target_daily = (
    df_parsed.dropna(subset=[TIMESTAMP_COLUMN])
    .set_index(TIMESTAMP_COLUMN)[TARGET_COLUMN]
    .resample("1D")
    .mean()
)

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(target_daily.index, target_daily.values, color="#dc2626", linewidth=0.8)
ax.set_title("Temperature over time (T, degC) — daily mean, downsampled from 10-min data for plotting")
ax.set_xlabel("Date")
ax.set_ylabel("T (degC)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(numeric_df[TARGET_COLUMN].dropna(), bins=80, color="#dc2626", alpha=0.8)
axes[0].set_title("Temperature distribution (T, degC)")
axes[0].set_xlabel("T (degC)")
axes[0].set_ylabel("Count (10-min observations)")

axes[1].boxplot(numeric_df[TARGET_COLUMN].dropna(), vert=True, showfliers=True,
                 boxprops=dict(color="#dc2626"), medianprops=dict(color="black"))
axes[1].set_title("Temperature boxplot (T, degC)")
axes[1].set_ylabel("T (degC)")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

target_robust_summary = numeric_df[TARGET_COLUMN].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
target_robust_summary

**Interpretation.** Temperature follows the expected strong annual cycle
(daily-mean plot) with no discontinuities coinciding with the temporal gaps
from §9. The distribution is unimodal and roughly symmetric with no
plausibility violations (§11 covers only `wv`/`max. wv`, not `T`).

## 15. Pressure Analysis – p (mbar)

In [ ]:
pressure_daily = (
    df_parsed.dropna(subset=[TIMESTAMP_COLUMN])
    .set_index(TIMESTAMP_COLUMN)["p (mbar)"]
    .resample("1D")
    .mean()
)

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(pressure_daily.index, pressure_daily.values, color="#0369a1", linewidth=0.8)
ax.set_title("Air pressure over time (p, mbar) — daily mean, downsampled for plotting")
ax.set_xlabel("Date")
ax.set_ylabel("p (mbar)")
plt.tight_layout()
plt.show()

**Interpretation.** Pressure oscillates within the plausible `[800, 1100]`
mbar range checked in §11 (zero violations), with no long-run drift or
step-changes that would suggest a sensor recalibration event.

## 16. Humidity Analysis – rh (%)

In [ ]:
humidity_daily = (
    df_parsed.dropna(subset=[TIMESTAMP_COLUMN])
    .set_index(TIMESTAMP_COLUMN)["rh (%)"]
    .resample("1D")
    .mean()
)

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(humidity_daily.index, humidity_daily.values, color="#059669", linewidth=0.8)
ax.axhline(100, color="gray", linestyle="--", linewidth=0.8, label="physical ceiling (100%)")
ax.set_title("Relative humidity over time (rh, %) — daily mean, downsampled for plotting")
ax.set_xlabel("Date")
ax.set_ylabel("rh (%)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**Interpretation.** Humidity stays within `[0, 100]%` (§11 confirms zero
violations) and shows the expected inverse seasonal relationship with
temperature (higher in winter, lower in summer).

## 17. Wind Analysis

Covers `wv (m/s)`, `max. wv (m/s)`, and `wd (deg)`. As established in §11,
`wv`/`max. wv` contain the `-9999` sentinel; the histograms below clip the
x-axis to the physically plausible range **for readability only** — the
sentinel rows are still counted explicitly and are not removed from the
underlying data.

In [ ]:
wv_violations = int(plausibility_df.set_index("column").loc["wv (m/s)", "total_violations"])
max_wv_violations = int(plausibility_df.set_index("column").loc["max. wv (m/s)", "total_violations"])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].hist(numeric_df["wv (m/s)"], bins=100, color="#7c3aed", alpha=0.85)
axes[0].set_xlim(0, 25)
axes[0].set_title(f"Wind speed distribution (wv, m/s)\nview clipped to [0,25]; {wv_violations} sentinel rows excluded from view")
axes[0].set_xlabel("wv (m/s)")
axes[0].set_ylabel("Count")

axes[1].hist(numeric_df["max. wv (m/s)"], bins=100, color="#9333ea", alpha=0.85)
axes[1].set_xlim(0, 25)
axes[1].set_title(f"Max wind speed distribution (max. wv, m/s)\nview clipped to [0,25]; {max_wv_violations} sentinel rows excluded from view")
axes[1].set_xlabel("max. wv (m/s)")
axes[1].set_ylabel("Count")

axes[2].hist(numeric_df["wd (deg)"], bins=72, color="#0891b2", alpha=0.85)
axes[2].set_title("Wind direction distribution (wd, deg)")
axes[2].set_xlabel("wd (deg)")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"wv (m/s) sentinel (<=-9000) rows      : {wv_violations}  ({wv_violations/report['row_count']*100:.4f}% of rows)")
print(f"max. wv (m/s) sentinel (<=-9000) rows : {max_wv_violations}  ({max_wv_violations/report['row_count']*100:.4f}% of rows)")

**Interpretation.** Excluding the sentinel rows from the *view*, wind speed
is right-skewed as expected (most readings are calm-to-moderate breeze,
long tail toward storms). Wind direction is non-uniform, with visible
preferred directions — the reason it must be treated as a **circular**
quantity rather than an ordinary numeric feature is demonstrated
quantitatively in §22.

## 18. Daily Seasonal Pattern

Mean temperature by hour-of-day, computed over the full raw dataset (this
is an aggregate of all 420k rows into 24 points, not a plot of raw points).

In [ ]:
hour_of_day = df_parsed[TIMESTAMP_COLUMN].dt.hour
by_hour = numeric_df[TARGET_COLUMN].groupby(hour_of_day).agg(["mean", "std"])

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(by_hour.index, by_hour["mean"], color="#dc2626", marker="o", markersize=3)
ax.fill_between(by_hour.index, by_hour["mean"] - by_hour["std"], by_hour["mean"] + by_hour["std"],
                 color="#dc2626", alpha=0.15, label="±1 std across all days")
ax.set_title("Mean temperature by hour of day (T, degC), all years pooled")
ax.set_xlabel("Hour of day (0-23)")
ax.set_ylabel("T (degC)")
ax.set_xticks(range(0, 24, 2))
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

**Interpretation.** Temperature follows a clear diurnal cycle — lowest in
the early morning, peaking mid-afternoon — which is the primary motivation
for the `hour_sin`/`hour_cos` cyclic time features planned in §23.

## 19. Monthly / Annual Seasonal Pattern

Mean temperature by calendar month, again aggregated over the full dataset.

In [ ]:
month = df_parsed[TIMESTAMP_COLUMN].dt.month
by_month = numeric_df[TARGET_COLUMN].groupby(month).agg(["mean", "std"])

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(by_month.index, by_month["mean"], color="#f97316", marker="o", markersize=4)
ax.fill_between(by_month.index, by_month["mean"] - by_month["std"], by_month["mean"] + by_month["std"],
                 color="#f97316", alpha=0.15, label="±1 std across years")
ax.set_title("Mean temperature by month (T, degC), 2009-2016 pooled")
ax.set_xlabel("Month")
ax.set_ylabel("T (degC)")
ax.set_xticks(range(1, 13))
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

**Interpretation.** A single, clean annual cycle (coldest in Jan/Feb,
warmest in Jul/Aug) with no anomalous months — motivates the
`dayofyear_sin`/`dayofyear_cos` features planned in §23 rather than a
categorical month feature, since day-of-year is continuous and periodic.

## 20. Correlation Analysis

Pearson correlation across raw numeric features (target included), full
dataset.

In [ ]:
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_columns)))
ax.set_xticklabels(numeric_columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(numeric_columns)))
ax.set_yticklabels(numeric_columns, fontsize=8)
for i in range(len(numeric_columns)):
    for j in range(len(numeric_columns)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6)
ax.set_title("Correlation heatmap — raw numeric features (Pearson r)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")
plt.tight_layout()
plt.show()

In [ ]:
NEAR_DUPLICATE_R_THRESHOLD = 0.995
target_corr = corr_matrix[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(key=np.abs, ascending=False)

pairs = []
cols = corr_matrix.columns.tolist()
for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        r = corr_matrix.loc[a, b]
        if abs(r) >= NEAR_DUPLICATE_R_THRESHOLD:
            pairs.append({"feature_a": a, "feature_b": b, "pearson_r": round(float(r), 5)})

near_duplicate_pairs = pd.DataFrame(pairs).sort_values("pearson_r", ascending=False)
print("Correlation of each feature with target T (degC):")
display(target_corr.round(3).to_frame("pearson_r_with_target"))
print(f"\nFeature pairs with |r| >= {NEAR_DUPLICATE_R_THRESHOLD}:")
near_duplicate_pairs

**Interpretation.** `T (degC)`, `Tpot (K)`, and `Tdew (degC)` are all
near-perfectly correlated — expected, since potential temperature and dew
point are thermodynamically derived from temperature and humidity. This
redundancy is noted (not acted on) and feeds directly into the feature
review in §21.

## 21. Accepted / Rejected Feature Review

No feature is rejected without evidence gathered above (§12 constant check,
§11 plausibility check, §20 redundancy check). "Review" means a caveat
exists that a modeling stage should be aware of; it does not remove the
feature from the schema.

In [ ]:
def feature_reason(column: str) -> tuple[str, str]:
    if column in report["constant_columns"]:
        return "Reject", "Validator flags this column as constant (nunique<=1); carries no signal."
    if column == "wd (deg)":
        return "Review", "Circular quantity (0deg==360deg); arithmetic mean is invalid (see Section 22)."
    if column in {"wv (m/s)", "max. wv (m/s)"}:
        return "Review", "Contains -9999 sentinel rows (Section 12); needs sentinel-to-missing handling before use."
    involved_in_near_dup = (
        (near_duplicate_pairs["feature_a"] == column) | (near_duplicate_pairs["feature_b"] == column)
    ).any() if not near_duplicate_pairs.empty else False
    if involved_in_near_dup:
        return "Review", f"|r| >= {NEAR_DUPLICATE_R_THRESHOLD} with another feature (Section 20); redundant but not identical."
    if column == TARGET_COLUMN:
        return "Keep", "Forecasting target."
    return "Keep", "Numeric, non-constant, no plausibility violations, not strongly redundant."


review_records = []
for column in numeric_columns:
    role = "Target" if column == TARGET_COLUMN else "Input feature"
    decision, reason = feature_reason(column)
    review_records.append({"feature": column, "role": role, "decision": decision, "reason": reason})

feature_review_df = pd.DataFrame(review_records)
feature_review_df

## 22. Hourly Aggregation Policy

The aggregation rule per feature is loaded from the approved
`tv2_data_cleaning_policy.json` and cross-checked against the raw schema.
This notebook records the policy but does not resample or write
`data/processed/`.


In [ ]:
cleaning_policy = json.loads(CLEANING_POLICY_PATH.read_text(encoding="utf-8"))
resampling_policy = cleaning_policy["resampling"]

mean_features = resampling_policy["mean_features"]
max_features = resampling_policy["max_features"]
circular_features = resampling_policy["circular_mean_features"]

assert set(mean_features) | set(max_features) | set(circular_features) == set(numeric_columns), (
    "Aggregation policy does not cover exactly the raw numeric columns"
)

agg_policy_df = pd.DataFrame(
    [{"feature": f, "aggregation": "mean"} for f in mean_features]
    + [{"feature": f, "aggregation": "max"} for f in max_features]
    + [{"feature": f, "aggregation": "circular_mean"} for f in circular_features]
).sort_values("feature").reset_index(drop=True)
print(f"Target frequency: {resampling_policy['frequency']}  (source: {CLEANING_POLICY_PATH.relative_to(PROJECT_ROOT)})")
agg_policy_df

**Why `wd (deg)` cannot use an arithmetic mean.** Wind direction is an angle
on a 360-degree circle, so `359°` and `1°` are only `2°` apart in reality,
but their arithmetic mean is `180°` — the exact opposite direction. The
correct aggregate is the **circular mean**:
`atan2(mean(sin(θ)), mean(cos(θ)))`. This is verified numerically below,
not just asserted.

## 23. Time Feature Engineering Plan

Create these features **after** hourly resampling:

- `hour_sin = sin(2π · hour / 24)` and `hour_cos = cos(2π · hour / 24)`
- `dayofyear_sin = sin(2π · day_of_year / 365.25)` and
  `dayofyear_cos = cos(2π · day_of_year / 365.25)`

Sin/cos pairs preserve the circular relationship between adjacent hours and
days. The formulas are recorded here; implementation belongs to TV2's
preprocessing module.


## 24. Missing-data Policy

Loaded verbatim from the approved `tv2_data_cleaning_policy.json` — this
notebook records and cross-checks the policy, it does not decide or apply
it.

In [ ]:
missing_policy = cleaning_policy["gap_policy"]["input_imputation"]
target_policy = {
    "target_imputation": cleaning_policy["gap_policy"]["target_imputation"],
    "drop_window_if_target_missing": cleaning_policy["gap_policy"]["drop_window_if_target_missing"],
}

policy_rows = [
    ("Input: split before fitting statistics", "Yes — median/scaler fit scope is train_only", "chronological split happens first (Section 25)"),
    ("Input: forward-fill", f"max {missing_policy['forward_fill_max_hours']} hour(s)", "short gaps only, causal"),
    ("Input: backward-fill", "Never" if not missing_policy["backward_fill"] else "Allowed", "prevents future leakage into past steps"),
    ("Input: residual fill", missing_policy["method"], f"median fit scope: {missing_policy['median_fit_scope']}"),
    ("Input: missing indicator", str(missing_policy["missing_indicator"]), "one binary flag per raw feature (Section 28 schema)"),
    ("Target: imputation", str(target_policy["target_imputation"]), "T (degC) is never imputed"),
    ("Target: window rule", f"drop_if_missing_in_horizon={target_policy['drop_window_if_target_missing']}", "any missing target in the 72h horizon drops that window"),
    ("NEW — wind sentinel (-9999)", "Not yet in approved policy", "Section 11/17 finding: must be mapped to missing before imputation, required follow-up (the final follow-up)"),
]
missing_policy_df = pd.DataFrame(policy_rows, columns=["Rule", "Value", "Note"])
missing_policy_df

## 25. Temporal Split Policy

Boundaries are computed from the **actual** `date_min`/`date_max` returned
by `validate_csv` (§8), split by elapsed time (not by an assumed row
count), matching `train_fraction/validation_fraction/test_fraction` in the
approved policy. Row counts shown are on the **raw 10-minute** data for
informational cross-checking only — the same time boundaries will be
re-applied to the hourly-resampled index once `src/data/preprocessing.py`
is implemented, so hourly row counts will differ slightly from these.

In [ ]:
split_policy = cleaning_policy["split_policy"]
assert split_policy["method"] == "chronological"

date_min = pd.Timestamp(str(ts_report["date_min"]))
date_max = pd.Timestamp(str(ts_report["date_max"]))
total_span = date_max - date_min

train_end = date_min + total_span * split_policy["train_fraction"]
validation_end = train_end + total_span * split_policy["validation_fraction"]
test_end = date_max

boundaries = pd.DataFrame(
    [
        {"split": "train", "start": date_min, "end": train_end},
        {"split": "validation", "start": train_end, "end": validation_end},
        {"split": "test", "start": validation_end, "end": test_end},
    ]
)

valid_ts = df_parsed[TIMESTAMP_COLUMN].dropna()
boundaries["raw_row_count"] = boundaries.apply(
    lambda r: int(((valid_ts >= r["start"]) & (valid_ts < r["end"] if r["split"] != "test" else valid_ts <= r["end"])).sum()),
    axis=1,
)
boundaries["pct_of_raw_rows"] = (boundaries["raw_row_count"] / len(valid_ts) * 100).round(2)
boundaries

In [ ]:
gap_edges = pd.to_datetime(gap_df[["before", "after"]].to_numpy().ravel()) if not gap_df.empty else pd.DatetimeIndex([])
for _, row in boundaries.iterrows():
    if row["split"] == "test":
        continue
    boundary_time = row["end"]
    if len(gap_edges) > 0:
        nearest_gap_hours = float(np.min(np.abs((gap_edges - boundary_time).total_seconds()))) / 3600
    else:
        nearest_gap_hours = float("inf")
    flag = "review" if nearest_gap_hours < INPUT_WINDOW_HOURS else "ok"
    print(f"{row['split']}->{'validation' if row['split']=='train' else 'test'} boundary "
          f"{boundary_time} is {nearest_gap_hours:.1f}h from the nearest known gap edge [{flag}]")

**Interpretation.** Split boundaries are derived purely from elapsed
calendar time between `date_min` and `date_max`, never from a random
permutation of rows — this by construction rules out "random time split" as
a leakage risk (§27). The boundary-to-gap distance check above flags
whether either critical gap from §9 sits close enough to a split edge to
require special handling when the sliding window (§26) is actually built.

## 26. Sliding-window Contract

The production dataset must produce:

- `X`: `[168, n_features]`
- `y`: `[72, 1]`
- input timestamps: `[168]`
- target timestamps: `[72]`

Each 240-hour span must remain inside one split and must not cross a known
timestamp gap. A window with a missing target is dropped; target values are
never imputed.


**Interpretation.** Per §26/§27's window policy (`same_split_only`,
`do_not_cross_timestamp_gap` in `tv2_data_cleaning_policy.json`), a window
generator must only place a window's 240 total hours (168 input + 72
horizon) fully inside one split, and must reject any window whose 240-hour
span crosses one of the gaps found in §9.

## 27. Leakage Guardrails

The approved policy requires chronological splitting, train-only scaler and
median fitting, causal forward-fill only, no backward-fill, no target
imputation, and windows confined to one split and one continuous time span.
The notebook records these rules; enforcement belongs to the preprocessing
and windowing implementation.


## 28. Locked Feature Schema Cross-check

The locked schema is cross-checked against the validator's raw feature order
and the planned cyclical-time and missing-indicator features. This cell does
not create a second schema artifact.


In [ ]:
locked_schema = json.loads(LOCKED_SCHEMA_PATH.read_text(encoding="utf-8"))
locked_raw_numeric = locked_schema["feature_origin"]["raw_numeric"]
locked_missing_indicators = locked_schema["feature_origin"]["missing_indicators"]
cyclic_time_features = ["hour_sin", "hour_cos", "dayofyear_sin", "dayofyear_cos"]

assert locked_raw_numeric == numeric_columns
assert len(locked_missing_indicators) == len(numeric_columns)

schema_check = pd.DataFrame(
    [
        ("Raw feature order matches validator", locked_raw_numeric == numeric_columns),
        ("Cyclical time features listed", cyclic_time_features == [
            "hour_sin", "hour_cos", "dayofyear_sin", "dayofyear_cos"
        ]),
        ("Target index", locked_schema["target"]["index"]),
        ("Feature count", locked_schema["n_features"]),
        ("Schema hash present", bool(locked_schema.get("schema_hash"))),
    ],
    columns=["check", "value"],
)
schema_check


## 29. Data Quality Gate Summary

Every gate below is computed from values already produced earlier in this
notebook, not restated from memory.

In [ ]:
gates = []

gates.append(("CSV readable", "PASS", "df_raw loaded successfully with pandas.read_csv."))
gates.append(("Schema", "PASS" if report["schema_valid"] else "FAIL",
              "Columns match EXPECTED_COLUMNS exactly." if report["schema_valid"] else "Fix column mismatch before proceeding."))
gates.append(("Target", "PASS" if report["target_column_exists"] else "FAIL",
              "T (degC) present." if report["target_column_exists"] else "Target column missing."))
gates.append(("Timestamp parsing", "PASS" if ts_report["parse_errors"] == 0 else "FAIL",
              "0 parse errors." if ts_report["parse_errors"] == 0 else f"{ts_report['parse_errors']} unparseable timestamps."))
gates.append(("Missing raw", "PASS" if report["total_missing_values"] == 0 else "WARNING",
              "No NaN cells in raw CSV." if report["total_missing_values"] == 0 else f"{report['total_missing_values']} missing cells."))
gates.append(("Numeric integrity", "PASS" if sum(report["non_numeric_by_column"].values()) == 0 else "FAIL",
              "All feature columns coerce cleanly to numeric."))
gates.append(("Infinite values", "PASS" if sum(report["infinite_by_column"].values()) == 0 else "FAIL",
              "No +/-inf values."))
gates.append(("Domain plausibility", "FAIL" if plausibility_df["total_violations"].sum() > 0 else "PASS",
              f"{int(plausibility_df['total_violations'].sum())} sentinel/implausible values (wv, max. wv); map to missing before imputation."))
gates.append(("Timestamp ordering", "WARNING" if not ts_report["monotonic_in_file"] else "PASS",
              "Raw file not strictly time-ordered; approved policy is sort-before-processing (cleaning policy)."))
gates.append(("Duplicate timestamps", "WARNING" if ts_report["duplicate_rows_after_first"] > 0 else "PASS",
              f"{ts_report['duplicate_rows_after_first']} duplicate rows, {n_conflicting_groups} conflicting groups; keep-first policy approved."))
gates.append(("Temporal gaps", "WARNING" if ts_report["gap_count"] > 0 else "PASS",
              f"{ts_report['gap_count']} gaps found; windows must not cross them (Section 26/27)."))
gates.append(("Ready for preprocessing", "WARNING",
              "Policy is recorded; preprocessing and split implementation remain to be completed."))

gate_df = pd.DataFrame(gates, columns=["Gate", "Status", "Required Action / Note"])

if (gate_df["Status"] == "FAIL").any():
    overall_gate = "FAIL"
elif (gate_df["Status"] == "WARNING").any():
    overall_gate = "WARNING"
else:
    overall_gate = "PASS"

print("Overall Data Quality Gate:", overall_gate)
gate_df